In [69]:
words = open('names.txt', 'r').read().splitlines()

In [70]:
import random
import torch
import torch.nn.functional as F

In [88]:
words = random.sample(words, len(words))

In [89]:
alphabets = ['.'] + sorted(list(set(''.join(words))))

stoi = {s: i for i, s in enumerate(alphabets)}
itos = {i: s for s, i in stoi.items()}
btoi = {}
n = 0
for ch1 in alphabets:
  for ch2 in alphabets:
    btoi[f'{ch1}{ch2}'] = n
    n += 1
itob = {i:s for s, i in btoi.items()}

In [90]:
# data split
train_words = words[:int(len(words)*0.8)]
dev_words = words[int(len(words)*0.8): int(len(words)*0.8) + int(len(words)*0.1)]
test_words = words[int(len(words)*0.8) + int(len(words)*0.1):]

In [99]:
# create dataset for bigram with train_words
xs, ys = [], []
for word in train_words:
  chs = ['.'] + list(word) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    xs.append(ix1)
    ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print(f'Number of examples: ', num)

# initialize 'network'
g = torch.Generator().manual_seed(2147483648)
w = torch.randn((27, 27), generator=g, requires_grad=True)

Number of examples:  182503


In [100]:
regularization_strength = 0.001

In [101]:
# gradient decent with train_data
for k in range(100):

  # forward pass
  xenc = F.one_hot(xs, num_classes=27).float()
  logits = (xenc @ w) # log-counts
  counts = logits.exp() # equivalent to N in the previous model
  probs = counts / counts.sum(1, keepdim=True) # normalizing each row
  loss = -(probs[torch.arange(num), ys]).log().mean() + regularization_strength*(w**2).mean()

  # backward pass
  w.grad = None
  loss.backward()

  # update
  w.data += -50 * w.grad

In [102]:
# testing loss data set of train_words

In [103]:
train_loss = None

for k in range(100):

  # forward pass
  xenc = F.one_hot(xs, num_classes=27).float()
  logits = (xenc @ w) # log-counts
  counts = logits.exp() # equivalent to N in the previous model
  probs = counts / counts.sum(1, keepdim=True) # normalizing each row
  loss = -(probs[torch.arange(num), ys]).log().mean()
  train_loss = loss
print(f'{train_loss=} at {regularization_strength=}')

train_loss=tensor(2.4717, grad_fn=<NegBackward0>) at regularization_strength=0.001


In [104]:
# testing loss data set of dev_words

In [105]:
# create dataset for bigram with dev_words
xs, ys = [], []
for word in dev_words:
  chs = ['.'] + list(word) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    xs.append(ix1)
    ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print(f'Number of examples: ', num)

Number of examples:  22915


In [106]:
dev_loss = None
for k in range(100):

  # forward pass
  xenc = F.one_hot(xs, num_classes=27).float()
  logits = (xenc @ w) # log-counts
  counts = logits.exp() # equivalent to N in the previous model
  probs = counts / counts.sum(1, keepdim=True) # normalizing each row
  loss = -(probs[torch.arange(num), ys]).log().mean()
  dev_loss = loss
print(f'{dev_loss=} at {regularization_strength=}')

dev_loss=tensor(2.4790, grad_fn=<NegBackward0>) at regularization_strength=0.001


In [26]:
# Loss noted
# At RS = 0; train_loss = 2.4723; dev_loss = 2.4782
# At RS = 0.001; train_loss = 2.4717; dev_loss = 2.4790
# At RS = 0.01; train_loss = 2.4748; dev_loss = 2.4748
# At RS = 0.1; train_loss = 2.5054; dev_loss = 2.5026
# At RS = 1; train_loss = 2.7202; dev_loss = 2.7260